# 233 — Condition-CONCATENATED clustering

Every other clustering notebook (210 raw · 212 rawds · 231 minus101 · 232 hg) treats
**one sample = one electrode × condition**, so the dominant structure available to find is
the *condition / modality* axis. This notebook builds the complementary view:

> **one sample = one ELECTRODE**, features = `[audio | picture | reading]` stitched in time

which asks **"what functional *type* is this contact across the whole task?"** rather than
*"which condition is this response?"*.

This is the **same sample construction** the stage-03 **parcellation** decoding and the
stage-04 **pooling** already use — so clusters from this track are directly comparable to
those results.

### Sample filter (mirrors 03's parcellation task)
An electrode is kept iff it

1. has **all three conditions** present, and
2. is **high-activity in ≥ 1** of them.

The gate is applied at the **electrode** level (not per sample), so a contact that only
responds in one condition still contributes its full three-condition profile.

### Feature sets
| feature_set | grid | dims | note |
|---|---|---|---|
| `concat_hg` | 1 × 900 | 900 | HG line (70–150 Hz) per condition, stitched — comparable to 232 |
| `concat_rawds` | 15 × 90 | 1350 | **the exact grid stage-04 pooling matches roles on** |
| `concat_raw` | 129 × 900 | 116 100 | full-resolution baseline (expect curse-of-dimensionality) |

Each is fed to the same `lf_cluster_run.fit_and_save` orchestrator, so outputs land in
`outputs/clustering/{method}/{feature_set}/runs/<id>/` and are picked up automatically by
`index.json`, **MOBA**, 211 validation and 252 recon.

> ⚠️ **213 ranking:** the *condition-selectivity* axis is meaningless here (every sample
> spans all three conditions) and degrades to `n_conditions=0`. Drop or replace that axis
> when ranking a `concat_*` run.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions import lf_concat as CC
from functions import lf_cluster_run as R

SCRIPT_NAME = '233_concat_clustering.ipynb'


In [ ]:
# ── data input ───────────────────────────────
INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── sample filter ────────────────────────────
CONDITIONS            = ('audio', 'picture', 'reading')
REQUIRE_HIGH_ACTIVITY = True     # keep electrode iff high-activity in >=1 condition

# ── representation ───────────────────────────
HG_BAND = (70.0, 150.0)
FMAX    = 500.0

# ── clustering ───────────────────────────────
# Wider floor than 232 ([6..20]): this is a new representation, so don't assume where
# the silhouette optimum sits. Read the silhouette-vs-K curve before trusting best_k.
K_RANGE      = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
HC_METHOD    = 'ward'
HC_METRIC    = 'euclidean'
RANDOM_STATE = cfg.RANDOM_STATE

# concat_raw is 116k features — set False to skip it on a quick pass.
RUN_CONCAT_RAW = True

print('conditions   :', CONDITIONS, '| gate high-activity>=1:', REQUIRE_HIGH_ACTIVITY)
print('K_RANGE      :', K_RANGE)


## 1 — Build the concatenated dataset

`build_concat_dataset` loads the **ungated** canonical ERSPs (cached separately in
`outputs/_dataset/concat_source/`), groups them by electrode, and applies the two-part
filter above. `X_concat` is `(n_electrodes, 129, 3×300)`.


In [ ]:
df_contacts, X_concat = CC.build_concat_dataset(
    INPUT_DIR, conditions=CONDITIONS, require_high_activity=REQUIRE_HIGH_ACTIVITY)

print(f'\nconcat dataset: {len(df_contacts)} electrodes · X_concat={X_concat.shape}')
print('patients:', df_contacts['patient_id'].nunique())
df_contacts.head()


In [ ]:
# How many conditions is each contact actually responsive in? (1 = modality-specific,
# 3 = responds across the board). Useful context for interpreting the clusters.
ax = df_contacts['n_high_activity'].value_counts().sort_index().plot.bar(
    figsize=(5, 3), color='#41ab5d')
ax.set_xlabel('# conditions the contact is high-activity in')
ax.set_ylabel('# electrodes'); ax.set_title('Responsiveness spread')
plt.tight_layout(); plt.show()


## 2 — Feature matrices

Three representations of the same concatenated map. `concat_rawds` downsamples **per
condition block** (never across the seam) with the same routine stage-04 pooling uses,
so its grid is identical to the pooling DS grid.


In [ ]:
X_hg    = CC.concat_hg_features(X_concat, hg_band=HG_BAND, fmax=FMAX)
X_rawds = CC.concat_rawds_features(X_concat, n_blocks=len(CONDITIONS), fmax_hz=FMAX)
print('concat_hg   :', X_hg.shape)
print('concat_rawds:', X_rawds.shape)

if RUN_CONCAT_RAW:
    X_raw = CC.concat_raw_features(X_concat)
    print('concat_raw  :', X_raw.shape, f'({X_raw.nbytes/1e6:.0f} MB)')


## 3 — Cluster

Each feature set × {K-Means, Ward}. `fit_and_save` writes the run dir, registers it in
`index.json`, and renders the standard figure set (silhouette knife + PCA, silhouette-vs-K,
centroids, similarity/centroid-distance heatmaps).


In [ ]:
# ── concat_hg ────────────────────────────────
m = R.fit_and_save(
    X_hg, df_keep=df_contacts, method='kmeans', feature_set='concat_hg',
    params={'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means', feature_set_label='Concatenated HG [a|p|r]',
    feature_names=CC.concat_feature_names('concat_hg'), notebook=SCRIPT_NAME)
print('best K (kmeans/concat_hg):', m['summary']['best_k'])

m = R.fit_and_save(
    X_hg, df_keep=df_contacts, method='hierarchical', feature_set='concat_hg',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': K_RANGE},
    method_label='Hierarchical', feature_set_label='Concatenated HG [a|p|r]',
    feature_names=CC.concat_feature_names('concat_hg'), notebook=SCRIPT_NAME)
print('best K (hc/concat_hg):', m['summary']['best_k'])


In [ ]:
# ── concat_rawds (same grid as stage-04 pooling) ──
m = R.fit_and_save(
    X_rawds, df_keep=df_contacts, method='kmeans', feature_set='concat_rawds',
    params={'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means', feature_set_label='Concatenated 15-band [a|p|r]',
    feature_names=CC.concat_feature_names('concat_rawds'), notebook=SCRIPT_NAME)
print('best K (kmeans/concat_rawds):', m['summary']['best_k'])

m = R.fit_and_save(
    X_rawds, df_keep=df_contacts, method='hierarchical', feature_set='concat_rawds',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': K_RANGE},
    method_label='Hierarchical', feature_set_label='Concatenated 15-band [a|p|r]',
    feature_names=CC.concat_feature_names('concat_rawds'), notebook=SCRIPT_NAME)
print('best K (hc/concat_rawds):', m['summary']['best_k'])


In [ ]:
# ── concat_raw (full resolution baseline) ─────
if RUN_CONCAT_RAW:
    m = R.fit_and_save(
        X_raw, df_keep=df_contacts, method='kmeans', feature_set='concat_raw',
        params={'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
        method_label='K-Means', feature_set_label='Concatenated full-res [a|p|r]',
        notebook=SCRIPT_NAME)
    print('best K (kmeans/concat_raw):', m['summary']['best_k'])

    m = R.fit_and_save(
        X_raw, df_keep=df_contacts, method='hierarchical', feature_set='concat_raw',
        params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': K_RANGE},
        method_label='Hierarchical', feature_set_label='Concatenated full-res [a|p|r]',
        notebook=SCRIPT_NAME)
    print('best K (hc/concat_raw):', m['summary']['best_k'])
else:
    print('[skip] RUN_CONCAT_RAW = False')


## 4 — Next

- **211** — consensus stability / gap / anatomy on these runs (all X-agnostic, works as-is).
- **213** — ranking: **drop the condition-selectivity axis** for `concat_*` (it is
  degenerate here) or replace it with a cross-condition consistency score.
- **252** — recon. Cleaner for this track: one row per electrode, so no duplicate contacts.
- **MOBA** — the runs appear in the dropdown automatically once `index.json` + `labels.csv`
  are committed. `condition` is set to `'concat'` so the condition view renders one category.
- **Compare** against the stage-04 pooling roles: `concat_rawds` uses the identical grid,
  so a cluster ↔ role cross-tab is apples-to-apples.
